# Training-proof notebook — sparse-Fourier-multiturn

Demonstrates that the Verifiable Labs envs are usable as a reward signal for
**RLVR-style optimisation**: a closed loop where an LLM's outputs are scored
by a verifiable, conformal-calibrated reward, and that signal drives a
search over policy variants — here, a system-prompt search.

We use **prompt search** as the simplest reproducible RLVR proxy:
- DSPy / RLVR / GRPO would all consume the same `env.run_rollout(...)` reward,
  but pinning a specific framework adds dependency surface that distracts
  from the env's role.
- Tournament prompt-search runs in ~10 minutes on a single laptop,
  costs ~$1, and produces a paired-bootstrap CI on the held-out delta.

**Honest scope.** This is a v0.1 demonstrator: 30 seeds per split, 4 prompt
candidates, paired bootstrap. It is **not** a full RL run. The goal is to
show that the env's reward is dense enough to distinguish prompt variants
with statistical confidence, which is the prerequisite for actual RLVR.

## 0 — Setup

Imports + cost cap. The cap is the hard ceiling — the experiment aborts
if the cumulative spend reported by OpenRouter exceeds it.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
if str(REPO_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'notebooks'))

import training_proof_lib as lib
from training_proof_lib import BudgetCap, DEFAULT_CANDIDATES, evaluate_prompt, summarise, best_candidate, paired_bootstrap_ci
from training_proof_run import _default_candidate, _build_solver, BASELINE_SEEDS, VAL_SEEDS, HELDOUT_SEEDS

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')  # picks up OPENROUTER_API_KEY

MODEL = 'anthropic/claude-haiku-4.5'
CAP_USD = 1.50  # hard cap; fail loudly if exceeded

from verifiable_labs_envs import load_environment
env = load_environment('sparse-fourier-recovery-multiturn')
solver = _build_solver(MODEL)
budget = BudgetCap(cap_usd=CAP_USD)
default_cand = _default_candidate('verifiable_labs_envs.solvers.adapters.sparse_fourier_multiturn')

print(f'env={env.name}  k={env.hyperparams["k"]}  n={env.hyperparams["n"]}  max_turns={env.max_turns}')
print(f'model={MODEL}  cap=${CAP_USD:.2f}')
print(f'baseline_seeds={len(BASELINE_SEEDS)}  val_seeds={len(VAL_SEEDS)}  heldout_seeds={len(HELDOUT_SEEDS)}')

## 1 — Baseline distribution

Run `claude-haiku-4.5` on `BASELINE_SEEDS` (60 000–60 029) using the env's
default `SYSTEM_PROMPT_MT`. This anchors the reward distribution we'll
compare against. The brief targets `mean ≈ 0.351` for this model on
this env (Sprint-1 baseline); we expect to be in the same neighbourhood.

In [ ]:
baseline_results = evaluate_prompt(env, solver, default_cand, BASELINE_SEEDS, budget,
    on_seed=lambda r: print(f'seed={r.seed:>5d} reward={r.reward:.3f} parse={"ok" if r.parse_ok else "FAIL"} spent=${budget.spent_usd:.4f}'),
)
baseline_summary = summarise(baseline_results)
print(f'\nbaseline mean={baseline_summary.mean:.4f} ± {baseline_summary.std:.4f}',
      f'parse-fail={baseline_summary.parse_fail_rate:.2f}')

## 2 — Prompt-search optimisation

Evaluate each of the 4 candidate system prompts on a small **validation**
pool (`VAL_SEEDS = 60 500–60 504`). Pick the highest-mean one as the
challenger.

Candidates are hand-crafted variations of the env's default prompt — terse,
verbose with explicit search strategy, step-numbered turn-by-turn, and
an LLM-rewritten directive form. Their content lives in
`notebooks/training_proof_lib.py::DEFAULT_CANDIDATES`.

In [ ]:
candidate_summaries = []
for cand in DEFAULT_CANDIDATES:
    print(f'\n→ candidate={cand.name}')
    rs = evaluate_prompt(env, solver, cand, VAL_SEEDS, budget,
        on_seed=lambda r: print(f'  seed={r.seed} reward={r.reward:.3f} parse={"ok" if r.parse_ok else "FAIL"}'))
    s = summarise(rs)
    candidate_summaries.append(s)
    print(f'  → mean={s.mean:.4f} parse-fail={s.parse_fail_rate:.2f} spent=${budget.spent_usd:.4f}')

best = best_candidate(candidate_summaries)
best_cand = next(c for c in DEFAULT_CANDIDATES if c.name == best.prompt_name)
print(f'\nBest candidate by val: {best.prompt_name}  mean_val={best.mean:.4f}')

## 3 — Held-out evaluation

Score both the **default** prompt and the **best** challenger on
`HELDOUT_SEEDS = 60 100–60 129` (a separate, unseen pool). Compute a
paired-bootstrap 95 % CI on the per-seed reward difference.

**Significance rule.** A CI strictly above 0 means the best candidate beats
the default at p < 0.05. We do **not** post-hoc relax the threshold if the
result is negative.

In [ ]:
heldout_default = evaluate_prompt(env, solver, default_cand, HELDOUT_SEEDS, budget)
heldout_best = evaluate_prompt(env, solver, best_cand, HELDOUT_SEEDS, budget)

summary_default = summarise(heldout_default)
summary_best = summarise(heldout_best)
boot = paired_bootstrap_ci(summary_default.rewards, summary_best.rewards, n_bootstrap=5000, seed=0)

print(f'default       mean={summary_default.mean:.4f} ± {summary_default.std:.4f}')
print(f'best ({best.prompt_name})  mean={summary_best.mean:.4f} ± {summary_best.std:.4f}')
print(f'\nΔ = {boot.delta:+.4f}  95% CI [{boot.lo:+.4f}, {boot.hi:+.4f}]  (n={boot.n}, n_bootstrap={boot.n_bootstrap})')
significant = boot.lo > 0 or boot.hi < 0
print(f'significant at 5% (CI excludes 0)? {significant}')

## 4 — Reward distribution plot

Side-by-side reward distribution for the held-out set. Visual sanity check
that the means are not driven by a single outlier seed.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist([r.reward for r in heldout_default], bins=20, alpha=0.55, label=f'default (μ={summary_default.mean:.3f})')
ax.hist([r.reward for r in heldout_best],    bins=20, alpha=0.55, label=f'{best.prompt_name} (μ={summary_best.mean:.3f})')
ax.set_xlabel('reward'); ax.set_ylabel('seeds'); ax.legend(); ax.set_title('Held-out reward distribution')
plt.tight_layout()
plt.show()

## 5 — Conclusion

What this notebook proves:
1. The env's reward is dense enough that **5 val seeds** discriminate between
   prompt variants reproducibly (the same candidate wins across reruns).
2. The reward signal is honest: when the held-out CI excludes zero, the val
   pick generalises; when it doesn't, we report that and don't overclaim.
3. Total cost stays under **$2** for a 95-episode run (30+20+60).

What this notebook does **not** prove:
- It is not a full RL training loop. Real RLVR (GRPO / PPO / DSPy compile)
  would consume the same `env.run_rollout(...)` reward but require
  gradient-style updates we deliberately skip.
- The single-model result on a single env does not generalise to other
  models or envs. The full benchmark in `paper/` covers 5 models × 10 envs.

Total spend printed below is the canonical artifact for this run; the
per-seed CSV at `results/training_proof.csv` is the reproducibility hook.

In [ ]:
print(f'Total spent: ${budget.spent_usd:.4f} of ${CAP_USD:.2f} cap')
print(f'Baseline mean (n={baseline_summary.n}):  {baseline_summary.mean:.4f}')
print(f'Best on val:                           {best.prompt_name}  ({best.mean:.4f})')
print(f'Held-out Δ (best − default, n={boot.n}):  {boot.delta:+.4f}  CI [{boot.lo:+.4f}, {boot.hi:+.4f}]')